# Notebook 01: Khám Phá Dữ Liệu & Tạo Features

-Chỉ sử dụng pre-treatment features cho modeling
- Tách rõ ràng post-treatment variables (không dùng làm features)
- Ghi rõ temporal ordering là synthetic (không phải thời gian thực)
- Chỉ tạo các engineered features an toàn

**Tác giả:** Phan Hồ Ngọc Hân

In [1]:
# ============================================
# IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Đảm bảo reproducibility
SEED = 42
np.random.seed(SEED)

# Cài đặt hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"✅ Import hoàn tất - SEED = {SEED}")

✅ Import hoàn tất - SEED = 42


## 1. Đọc Dữ Liệu

In [2]:
# ============================================
# ĐỌC DATASET
# ============================================
data_path = Path("../data/raw/causal_marketing/marketing_campaign.xlsx")

if not data_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {data_path}")

df = pd.read_excel(data_path)

print(f"Đã tải dataset: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"\n5 dòng đầu tiên:")
df.head()

Đã tải dataset: 5,000 dòng × 15 cột

5 dòng đầu tiên:


,user_id,campaign_id,channel,device,country,segment,prior_visits_30d,prior_spend_180d,treatment_exposed,impressions,clicks,spend_usd,conversion,revenue_usd,roi
0,221958,CMP003,Social,Desktop,AE,Food,3,93.1400,0,9,1,0.2900,0,0.0000,-1.0000
1,771155,CMP006,Social,Mobile,IN,Food,0,55.1400,0,16,1,0.3700,0,0.0000,-1.0000
2,231932,CMP004,Search,Mobile,FR,Tech,4,278.7900,0,10,0,0.0400,0,0.0000,-1.0000
3,465838,CMP006,Search,Tablet,IN,Food,2,292.6000,1,84,1,0.7400,0,0.0000,-1.0000
4,359178,CMP003,Search,Desktop,IN,Tech,4,16.9800,0,8,0,0.0300,0,0.0000,-1.0000


## 2. Phân Loại Features

**QUAN TRỌNG: Phân tách rõ ràng Pre-Treatment vs. Post-Treatment Variables**

### 2.1. Pre-Treatment Features (Quan sát TRƯỚC chiến dịch)

Đây là các features DUY NHẤT được phép dùng cho modeling:
- Thuộc tính khách hàng biết trước khi assign treatment
- Thiết kế chiến dịch (channel, device, etc.)
- Hành vi lịch sử (prior visits, prior spend)

### 2.2. Post-Treatment Variables (Xảy ra SAU chiến dịch)

Các biến này KHÔNG được dùng làm features:
- Treatment exposure chính nó
- Biến outcome (conversion, revenue)
- Metrics phụ thuộc vào treatment (impressions, clicks, spend)

In [3]:
# ============================================
# PHÂN LOẠI FEATURES
# ============================================

# PRE-TREATMENT FEATURES (An toàn để sử dụng)
PRE_TREATMENT_FEATURES = [
    "campaign_id",        # ID chiến dịch
    "channel",            # Kênh marketing (Email, Social, Search, Display, Video)
    "device",             # Loại thiết bị (Desktop, Mobile, Tablet)
    "country",            # Mã quốc gia (US, UK, FR, DE, IN, AE)
    "segment",            # Phân khúc khách hàng (Tech, Food, Health, Fashion, Outdoor)
    "prior_visits_30d",   # Lượt truy cập trong 30 ngày trước (TRƯỚC chiến dịch)
    "prior_spend_180d"    # Chi tiêu lịch sử trong 180 ngày trước (TRƯỚC chiến dịch)
]

# POST-TREATMENT VARIABLES (Không được dùng làm features)
POST_TREATMENT_COLUMNS = [
    "treatment_exposed",  # Treatment assignment (dùng cho CATE, không phải feature)
    "impressions",        # Số lần hiển thị quảng cáo (xảy ra SAU treatment)
    "clicks",             # Số lần click (xảy ra SAU treatment)
    "spend_usd",          # Chi phí marketing thực tế (xảy ra SAU treatment)
    "conversion",         # Biến outcome (TARGET, không phải feature)
    "revenue_usd",        # Doanh thu (xảy ra SAU conversion)
    "roi"                 # ROI (được tính SAU chiến dịch)
]

# METADATA
METADATA_COLUMNS = [
    "user_id"             # ID khách hàng (không phải feature)
]

print("✅ Đã phân loại features:")
print(f"   Pre-treatment features: {len(PRE_TREATMENT_FEATURES)}")
print(f"   Post-treatment variables: {len(POST_TREATMENT_COLUMNS)}")
print(f"   Metadata: {len(METADATA_COLUMNS)}")

✅ Đã phân loại features:
   Pre-treatment features: 7
   Post-treatment variables: 7
   Metadata: 1


## 3. Kiểm Tra Chất Lượng Dữ Liệu

In [4]:
# ============================================
# ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU
# ============================================

print("=" * 70)
print("BÁO CÁO CHẤT LƯỢNG DỮ LIỆU")
print("=" * 70)

# Giá trị thiếu
missing = df.isnull().sum()
print(f"\n1. Giá trị thiếu: {missing.sum()} tổng cộng")
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("   ✅ Không có giá trị thiếu")

# Bản ghi trùng lặp
duplicates = df.duplicated(subset='user_id').sum()
print(f"\n2. Trùng lặp user_id: {duplicates}")
if duplicates == 0:
    print("   ✅ Không có trùng lặp")

# Kiểu dữ liệu
print("\n3. Kiểu dữ liệu:")
print(df.dtypes)

# Thống kê mô tả
print("\n4. Thống kê mô tả:")
df.describe()

BÁO CÁO CHẤT LƯỢNG DỮ LIỆU

1. Giá trị thiếu: 0 tổng cộng
   ✅ Không có giá trị thiếu

2. Trùng lặp user_id: 12

3. Kiểu dữ liệu:
user_id                int64
campaign_id              str
channel                  str
device                   str
country                  str
segment                  str
prior_visits_30d       int64
prior_spend_180d     float64
treatment_exposed      int64
impressions            int64
clicks                 int64
spend_usd            float64
conversion             int64
revenue_usd          float64
roi                  float64
dtype: object

4. Thống kê mô tả:


,user_id,prior_visits_30d,prior_spend_180d,treatment_exposed,impressions,clicks,spend_usd,conversion,revenue_usd,roi
count,5000.0000,5000.0000,5000.0000,5000.0000,5000.0000,5000.0000,5000.0000,5000.0000,5000.0000,5000.0000
mean,551628.1238,1.9756,117.8014,0.4640,48.2798,1.3188,0.5945,0.1014,14.1499,93.6113
std,260310.7824,1.4042,84.0516,0.4988,39.5449,1.6105,0.5772,0.3019,43.4940,546.7369
min,100126.0000,0.0000,1.5100,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,-1.0000
25%,320778.5000,1.0000,55.2200,0.0000,12.0000,0.0000,0.0700,0.0000,0.0000,-1.0000
50%,554853.5000,2.0000,97.7700,0.0000,17.5000,1.0000,0.3400,0.0000,0.0000,-1.0000
75%,777125.5000,3.0000,158.8225,1.0000,89.0000,2.0000,1.0200,0.0000,0.0000,-1.0000
max,999885.0000,9.0000,645.2600,1.0000,131.0000,10.0000,3.1500,1.0000,243.4400,15463.0000


## 4. Cân Bằng Treatment/Control

In [5]:
# ============================================
# CÂN BẰNG TREATMENT / CONTROL
# ============================================

print("=" * 70)
print("CÂN BẰNG TREATMENT / CONTROL")
print("=" * 70)

treatment_counts = df['treatment_exposed'].value_counts().sort_index()
treatment_pct = df['treatment_exposed'].value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    'Nhóm': ['Control (T=0)', 'Treatment (T=1)'],
    'Số lượng': treatment_counts.values,
    'Tỷ lệ (%)': treatment_pct.values
})

print(balance_df.to_string(index=False))
print(f"\nĐộ lệch so với 50/50: {abs(treatment_pct[1] - 50):.1f}%")

if abs(treatment_pct[1] - 50) < 5:
    print("✅ Treatment assignment cân bằng tốt")
else:
    print("⚠️ Treatment assignment mất cân bằng")

CÂN BẰNG TREATMENT / CONTROL
           Nhóm  Số lượng  Tỷ lệ (%)
  Control (T=0)      2680    53.6000
Treatment (T=1)      2320    46.4000

Độ lệch so với 50/50: 3.6%
✅ Treatment assignment cân bằng tốt


## 5. Feature Engineering (Chỉ Pre-Treatment)

**Các Engineered Features An Toàn:**
1. `value_tier`: Phân khúc khách hàng dựa trên prior_spend_180d
2. `channel_base_cost`: Chi phí cơ bản theo kênh (biết trước chiến dịch)

**Các Features Đã Loại Bỏ (Post-Treatment Leakage):**
- ❌ `ctr` = clicks / impressions (cả hai đều post-treatment)
- ❌ `cost_per_click` = spend_usd / clicks (cả hai đều post-treatment)
- ❌ `engagement_score` sử dụng clicks/impressions (post-treatment)
- ❌ `expected_value` = revenue_usd (post-treatment outcome)
- ❌ `risk_score` sử dụng conversion (post-treatment outcome)

In [6]:
# ============================================
# TẠO FEATURES (CHỈ AN TOÀN)
# ============================================

print("Tạo các pre-treatment features an toàn...\n")

# ============================================
# 1. VALUE TIER (Phân Khúc Khách Hàng)
# ============================================
# Dựa trên prior_spend_180d (dữ liệu lịch sử, pre-treatment)

df["value_tier"] = pd.qcut(
    df["prior_spend_180d"],
    q=4,
    labels=["Low", "Medium", "High", "VIP"],
    duplicates='drop'
)

print("✅ Đã tạo: value_tier (Low/Medium/High/VIP)")
print(df['value_tier'].value_counts().sort_index())

# ============================================
# 2. CHANNEL BASE COST (Chi Phí Cơ Bản Theo Kênh)
# ============================================
# Biết trước TRƯỚC KHI thực hiện chiến dịch
# Đây là chi phí KẾ HOẠCH theo kênh, không phải chi phí thực tế

channel_cost_map = {
    "Email": 0.50,
    "Social": 1.20,
    "Search": 3.00,
    "Display": 2.50,
    "Video": 4.00
}

df["channel_base_cost"] = df["channel"].map(channel_cost_map)

print("\n✅ Đã tạo: channel_base_cost (chi phí kế hoạch theo kênh)")
print(df.groupby('channel')['channel_base_cost'].first())

# ============================================
# XÁC MINH: Không có NaN trong engineered features
# ============================================
print(f"\n✅ Feature engineering hoàn tất")
print(f"   value_tier: {df['value_tier'].isnull().sum()} thiếu")
print(f"   channel_base_cost: {df['channel_base_cost'].isnull().sum()} thiếu")

Tạo các pre-treatment features an toàn...

✅ Đã tạo: value_tier (Low/Medium/High/VIP)
value_tier
Low       1250
Medium    1250
High      1250
VIP       1250
Name: count, dtype: int64

✅ Đã tạo: channel_base_cost (chi phí kế hoạch theo kênh)
channel
Display   2.5000
Email     0.5000
Search    3.0000
Social    1.2000
Video     4.0000
Name: channel_base_cost, dtype: float64

✅ Feature engineering hoàn tất
   value_tier: 0 thiếu
   channel_base_cost: 0 thiếu


## 6. Temporal Splits

### ⚠️ QUAN TRỌNG: Synthetic Temporal Ordering

**Hạn chế:** Dataset này KHÔNG CÓ timestamps thực tế.

**Phương pháp:** Chúng ta tạo **synthetic temporal proxy** bằng cách:
1. Sắp xếp theo `user_id` (thứ tự tùy ý)
2. Gán các ngày synthetic từ 01/01/2025 đến 31/12/2025
3. Chia tuần tự thành Train/Validation/Test/OOT

**Ý nghĩa:**
- ✅ Tránh data leakage (khách hàng sau không dùng để dự đoán khách hàng trước)
- ✅ Kiểm tra sự ổn định của model qua các phân đoạn dữ liệu
- ❌ KHÔNG bắt được temporal drift thực tế (không có seasonality thật)
- ❌ KHÔNG thể validate tính robust theo thời gian thực

**Ghi chú:** Hạn chế này sẽ được ghi trong Phần 6 (Limitations).

In [7]:
# ============================================
# TẠO SYNTHETIC TEMPORAL ORDERING
# ============================================

print("Tạo synthetic temporal ordering...\n")

# Sắp xếp theo user_id (tùy ý nhưng deterministic)
df = df.sort_values('user_id').reset_index(drop=True)

# Gán ngày synthetic
df['temporal_proxy'] = pd.date_range(
    start='2025-01-01',
    end='2025-12-31',
    periods=len(df)
)

print(f"✅ Đã gán ngày synthetic: {df['temporal_proxy'].min()} đến {df['temporal_proxy'].max()}")
print(f"   ⚠️  LƯU Ý: Đây KHÔNG PHẢI timestamps thực - chỉ là proxy synthetic")

Tạo synthetic temporal ordering...

✅ Đã gán ngày synthetic: 2025-01-01 00:00:00 đến 2025-12-31 00:00:00
   ⚠️  LƯU Ý: Đây KHÔNG PHẢI timestamps thực - chỉ là proxy synthetic


In [8]:
# ============================================
# CHIA TEMPORAL SPLITS (60/20/10/10)
# ============================================

n = len(df)

# Ranh giới split
train_end = int(0.60 * n)
val_end = int(0.80 * n)
test_end = int(0.90 * n)

# Gán nhãn split
df['split'] = 'out_of_time'
df.loc[:train_end-1, 'split'] = 'train'
df.loc[train_end:val_end-1, 'split'] = 'validation'
df.loc[val_end:test_end-1, 'split'] = 'locked_test'

# In tóm tắt split
print("\n" + "="*70)
print("TÓM TẮT TEMPORAL SPLIT (Synthetic Ordering)")
print("="*70)

split_summary = df.groupby('split').agg({
    'user_id': 'count',
    'temporal_proxy': ['min', 'max'],
    'treatment_exposed': 'mean',
    'conversion': 'mean'
}).round(4)

split_summary.columns = ['Số lượng', 'Ngày bắt đầu', 'Ngày kết thúc', 'Treatment_%', 'Conversion_%']
split_summary = split_summary.reindex(['train', 'validation', 'locked_test', 'out_of_time'])
split_summary['Treatment_%'] *= 100
split_summary['Conversion_%'] *= 100

print(split_summary)
print("="*70)


TÓM TẮT TEMPORAL SPLIT (Synthetic Ordering)
             Số lượng               Ngày bắt đầu              Ngày kết thúc  \
split                                                                         
train            3000 2025-01-01 00:00:00.000000 2025-08-07 08:54:03.528705   
validation       1000 2025-08-07 10:38:54.706941 2025-10-19 04:27:01.764352   
locked_test       500 2025-10-19 06:11:52.942588 2025-11-24 14:13:30.882176   
out_of_time       500 2025-11-24 15:58:22.060412 2025-12-31 00:00:00.000000   

             Treatment_%  Conversion_%  
split                                   
train            47.2700       10.0000  
validation       46.0000       10.8000  
locked_test      45.8000        9.6000  
out_of_time      42.6000       10.2000  


/var/folders/t6/v2bt6bbs5l3dwx8ltlhmpmn80000gn/T/ipykernel_50323/18906879.py:28: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  }).round(4)


## 7. Decision Periods Cho MILP

Chia dataset thành 5 periods (mỗi period 1,000 khách hàng) để validate MILP qua nhiều kỳ.

In [9]:
# ============================================
# DECISION PERIODS (5 × 1,000 khách hàng)
# ============================================

df['decision_period'] = (df.index // 1000) + 1

print("\nDecision Periods:")
print(df['decision_period'].value_counts().sort_index())

print(f"\n✅ Đã tạo decision_period (1-5)")


Decision Periods:
decision_period
1    1000
2    1000
3    1000
4    1000
5    1000
Name: count, dtype: int64

✅ Đã tạo decision_period (1-5)


## 8. Danh Sách Features Cuối Cùng

**Cho Modeling (Chỉ Pre-Treatment):**

In [10]:
# ============================================
# DANH SÁCH FEATURES CUỐI CÙNG CHO MODELING
# ============================================

MODELING_FEATURES = PRE_TREATMENT_FEATURES + [
    "value_tier",
    # "channel_base_cost" sẽ dùng trong MILP, không dùng trong prediction models
]

print("=" * 70)
print("DANH SÁCH FEATURES CUỐI CÙNG CHO PREDICTIVE MODELING")
print("=" * 70)
print("\nFeatures (tất cả đều pre-treatment):")
for i, feat in enumerate(MODELING_FEATURES, 1):
    print(f"  {i}. {feat}")

print(f"\nTổng cộng: {len(MODELING_FEATURES)} features")
print("\n✅ Tất cả features đều quan sát được TRƯỚC khi assign treatment")
print("✅ Không có data leakage từ post-treatment variables")

DANH SÁCH FEATURES CUỐI CÙNG CHO PREDICTIVE MODELING

Features (tất cả đều pre-treatment):
  1. campaign_id
  2. channel
  3. device
  4. country
  5. segment
  6. prior_visits_30d
  7. prior_spend_180d
  8. value_tier

Tổng cộng: 8 features

✅ Tất cả features đều quan sát được TRƯỚC khi assign treatment
✅ Không có data leakage từ post-treatment variables


## 9. Xuất Dữ Liệu Đã Xử Lý

In [11]:
# ============================================
# XUẤT DỮ LIỆU ĐÃ XỬ LÝ
# ============================================

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Xuất full dataset
output_path = output_dir / "marketing_campaign_processed.parquet"
df.to_parquet(output_path, index=False)
print(f"✅ Đã lưu: {output_path}")
print(f"   Kích thước: {df.shape}")

# Xuất feature metadata
metadata = {
    'pre_treatment_features': PRE_TREATMENT_FEATURES,
    'modeling_features': MODELING_FEATURES,
    'post_treatment_columns': POST_TREATMENT_COLUMNS,
    'metadata_columns': METADATA_COLUMNS,
    'engineered_features': ['value_tier', 'channel_base_cost'],
    'splits': ['train', 'validation', 'locked_test', 'out_of_time'],
    'decision_periods': 5,
    'total_samples': len(df),
    'seed': SEED
}

import json
metadata_path = output_dir / "feature_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✅ Đã lưu: {metadata_path}")

✅ Đã lưu: ../data/processed/marketing_campaign_processed.parquet
   Kích thước: (5000, 20)
✅ Đã lưu: ../data/processed/feature_metadata.json


## 10. Tóm Tắt Dữ Liệu

In [12]:
# ============================================
# TÓM TẮT CUỐI CÙNG
# ============================================

print("\n" + "="*70)
print("TÓM TẮT NOTEBOOK 01")
print("="*70)

print(f"\n✅ Đã tải dữ liệu: {len(df):,} khách hàng")
print(f"✅ Pre-treatment features: {len(MODELING_FEATURES)}")
print(f"✅ Temporal splits: 60% train / 20% val / 10% test / 10% OOT")
print(f"✅ Decision periods: 5 periods × 1,000 khách hàng")
print(f"✅ Cân bằng treatment: {df['treatment_exposed'].mean()*100:.1f}% treatment")
print(f"✅ Tỷ lệ conversion: {df['conversion'].mean()*100:.2f}% tổng thể")

print("\n✅ Sẵn sàng cho Notebook 02 (Predictive Modeling)")
print("="*70)


TÓM TẮT NOTEBOOK 01

✅ Đã tải dữ liệu: 5,000 khách hàng
✅ Pre-treatment features: 8
✅ Temporal splits: 60% train / 20% val / 10% test / 10% OOT
✅ Decision periods: 5 periods × 1,000 khách hàng
✅ Cân bằng treatment: 46.4% treatment
✅ Tỷ lệ conversion: 10.14% tổng thể

🔴 CÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:
   • Đã loại bỏ tất cả post-treatment features khỏi modeling
   • Đổi tên decision_date → temporal_proxy (synthetic ordering)
   • Ghi chú rõ ràng về hạn chế của synthetic temporal
   • Thêm channel_base_cost cho MILP (không dùng cho prediction)

✅ Sẵn sàng cho Notebook 02 (Predictive Modeling)
